# 

By Edson Petry

In [1]:
import os
from pysnail import Dataset, qsmooth
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Needed to use older version of pandas due to error involving "squeeze" argument within PySnail
import pandas as pd
print(pd.__version__)

ModuleNotFoundError: No module named 'pysnail'

In [ ]:
RNA_metadata_df = pd.read_csv("in/metadata/RNA_metadata.tsv", sep='\t', index_col=0).reset_index(drop=True)

In [ ]:
RNA_metadata_df

In [ ]:
RNA_counts_df = pd.read_csv("in/raw_data/RNA/RNA_raw_counts.tsv", sep="\t")

In [ ]:
RNA_counts_df

In [ ]:
to_remove = pd.read_excel("in/raw_data/RNAseq Compendium Samples.xlsx")

In [ ]:
to_remove

In [ ]:
samples_to_remove = to_remove['Sample']
RNA_counts_df= RNA_counts_df.drop(columns=samples_to_remove)

In [ ]:
RNA_counts_log2_df = np.log2(RNA_counts_df.iloc[:, 1:] + 1)

In [ ]:
RNA_counts_log2_df

In [ ]:
def plotDens(df, ylabel='', xlabel='', title='Density Plot', amount=50):
    """
    Plots the density (KDE) plots for columns in a DataFrame.

    Parameters:
    df (pd.DataFrame): The DataFrame containing the data to plot.
    ylabel (str): Label for the y-axis. Default is an empty string.
    xlabel (str): Label for the x-axis. Default is an empty string.
    title (str): Title for the plot. Default is 'Density Plot'.
    amount (int): Number of columns to plot starting from the second column. Default is 50.

    Returns:
    None
    """
    plt.figure(figsize=(12, 8))

    for column in df.columns[1:amount]:
        sns.kdeplot(data=df[column], warn_singular=False)
    
    # Show the plot
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)

In [ ]:
plotDens(RNA_counts_log2_df, xlabel='log2 RNA counts', title= 'Density Plot for the First 50 Samples')

In [ ]:
plotDens(RNA_counts_log2_df, xlabel='log2 RNA counts', title= 'Density Plot for the All Samples', amount=4622)

In [ ]:
plotDens(RNA_counts_log2_df, xlabel='log2 RNA counts', title= 'Density Plot for first 200 Samples', amount=200)

## Quality assurance & TPM Calculation

### TPM Calculation
Utilzing rnanorm (https://pypi.org/project/rnanorm/1.5.1/) we are able to pass in the genome annotation (https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/195/955/GCF_000195955.2_ASM19595v2/) and get TPM (Transcripts Per Million) values for our RNA counts.
We convert RNA counts to TPM values in order to make accurate comparisons of gene expression levels across different samples and conditons.

#### Installation

In [ ]:
%pip install rnanorm

#### Formatting data

In [ ]:
RNA_counts_df2 = RNA_counts_df.set_index("Geneid")
RNA_counts_df2.index.name = None
RNA_counts_dfT = RNA_counts_df2.T

In [ ]:
RNA_counts_dfT

In [ ]:
RNA_counts_dfT.to_csv("in/tpm_conversion/counts")

Instructions for rnanorm
```
# Use GTF file
rnanorm tpm exp.csv --gtf annotations.gtf > exp_out.csv
# Use gene lengths file
rnanorm tpm exp.csv --gene-lengths lenghts.csv > exp_out.csv
# Example of gene lengths file
cat lenghts.csv
gene_id,gene_length
Gene_1,200
Gene_2,300
Gene_3,500
Gene_4,1000
Gene_5,1000
```

Loading in genome annotation file

In [ ]:
genome_annotation_df = pd.read_csv("in/tpm_conversion/annotation_file/GCF_000195955.2_ASM19595v2_feature_table.txt", sep="\t")

In [ ]:
genome_annotation_df

In [ ]:
gene_lengths = genome_annotation_df[["locus_tag", "feature_interval_length"]]
gene_lengths = gene_lengths.rename(columns={"locus_tag":"gene_id","feature_interval_length":"gene_length"})
gene_lengths = gene_lengths.drop_duplicates(subset='gene_id', keep='first').reset_index(drop=True)
gene_lengths

Here we filter our lengths table to only include the genes that we have in our RNA counts table

In [ ]:
filtered_gene_lengths = gene_lengths[gene_lengths['gene_id'].isin(RNA_counts_df['Geneid'])].reset_index(drop=True)
filtered_gene_lengths

In [ ]:
filtered_gene_lengths.to_csv("in/tpm_conversion/lengths/lenghts.csv", index=False)

#### Running rnanorm

In [ ]:
%pip install rnanorm

In [ ]:
%rnanorm tpm tpm_conversion/counts/TPM_conversion_input.csv --gene-lengths tpm_conversion/lengths/lenghts.csv > tpm_conversion/output/TPM_counts.csv

In [ ]:
RNA_tpm_counts_df = pd.read_csv("out/tpm_conversion_out/output/TPM_counts.csv")

In [ ]:
RNA_tpm_counts_df

In [ ]:
RNA_tpm_counts_dfa = RNA_tpm_counts_df.set_index("Unnamed: 0")
RNA_tpm_counts_dfa.index.name = None
RNA_tpm_counts_dfb = RNA_tpm_counts_dfa.T.reset_index().rename(columns={"index":"Geneid"})

In [ ]:
RNA_tpm_counts_dfb

#### Log2 TPM transformation

In [ ]:
TPM_log2_df = np.log2(RNA_tpm_counts_dfb.iloc[:, 1:] + 1)

In [ ]:
TPM_log2_df

In [ ]:
plotDens(TPM_log2_df, xlabel='TPM log2 RNA counts', title= 'Density Plot for the First 50 Samples')

In [ ]:
plotDens(TPM_log2_df, xlabel='TPM log2 RNA counts', title= 'Density Plot for the First 200 Samples', amount=200)

In [ ]:
plotDens(TPM_log2_df, xlabel='TPM log2 RNA counts', title= 'Density Plot for the All Samples', amount=4622)

### Filtering low quality samples

Dr. Yang determined that 444 samples were "low-quality" due to very low RNA counts values.

In [ ]:
exclude = pd.read_excel("in/raw_data/RNA/exclude/exclude.xlsx", header=None)
exclude.columns = ['Sample']

In [ ]:
samples_to_exclude = exclude['Sample'].tolist()

In [ ]:
print("Number of low-quality samples:", len(samples_to_exclude))

In [ ]:
TPM_log2_filtered_df = TPM_log2_df.drop(columns=samples_to_exclude)

In [ ]:
TPM_log2_filtered_df2 = pd.concat([RNA_tpm_counts_dfb['Geneid'], TPM_log2_filtered_df], axis=1)

In [ ]:
TPM_log2_filtered_df2

In [ ]:
plotDens(TPM_log2_filtered_df2, xlabel='TPM log2 RNA counts (filtered)', title= 'Density Plot for the First 50 Samples')

In [ ]:
plotDens(TPM_log2_filtered_df2, xlabel='TPM log2 RNA counts (filtered)', title= 'Density Plot for First 200 Samples', amount=200)

In [ ]:
plotDens(TPM_log2_filtered_df2, xlabel='TPM log2 RNA counts (filtered)', title= 'Density Plot for All Samples', amount=4178)

## Smooth quantile Normalization
Smooth quantile normalization (qsmooth) is an advanced normalization method used in genomic data analysis. It generalizes traditional quantile normalization by allowing the statistical distribution of each sample to be the same within biological groups, but different between groups. This approach helps in maintaining biological differences while reducing technical variability.

Here we use PySnail (https://github.com/kuijjerlab/PySNAIL/tree/main) to apply smooth quantile normalization to our dataset.

In [ ]:
def pysnail(expression_data_path, group_data_path, output_path):
    """
    Apply quantile normalization to expression data.

    This function reads expression data and group information from the provided
    file paths, performs quantile normalization on the expression data, and
    saves the normalized data to the specified output path.

    Parameters:
    expression_data_path (str): Path to the input expression data file in TSV format.
    group_data_path (str): Path to the input group data file in TSV format.
    output_path (str): Path to save the normalized expression data file in TSV format.

    Returns:
    pandas.DataFrame: DataFrame containing the normalized expression data.
    """
    xprs = os.path.realpath(expression_data_path)
    groups = os.path.realpath(group_data_path)
    output_file = os.path.realpath(output_path)

    dataset = Dataset(xprs, groups, **{'index_col': 0, 'sep':'\t'})
    xprs_norm, qstat = qsmooth(dataset, aggregation='auto', cutoff=0.15)

    xprs_norm.to_csv(output_file, sep='\t')
    print(f"Normalized expression data saved to {output_file}")
    
    return xprs_norm

I then loaded in the groupings data. The samples are grouped by biological replicates. These groups were assigned by myself and Oliver Gu. The groupings are for the quantile smooth normalization function that will be used after the raw counts are converted to TPM log2 values.

### Reading in groups

In [ ]:
RNA_groups_df = pd.read_csv("in/raw_data/RNA/groups/RNA_groups.tsv", sep="\t").rename(columns={"Group":"temp"})

In [ ]:
RNA_groups_df

In [ ]:
unique_values = RNA_groups_df["temp"].unique()
group_mapping = {value: idx + 1 for idx, value in enumerate(unique_values)}

# Apply the mapping to create the Group column
RNA_groups_df['Group'] = RNA_groups_df["temp"].map(group_mapping)

In [ ]:
RNA_groups_df

In [ ]:
RNA_groups_df = RNA_groups_df.drop(columns="temp")

In [ ]:
RNA_groups_df

### Writing data for qsmooth

In [ ]:
RNA_groups_df.to_csv("in/qsmooth_in/RNA/groups/RNA_groups_final.tsv", sep='\t', index=False, header=None)
TPM_log2_filtered_df2.to_csv("in/qsmooth_in/RNA/xprs/RNA_xprs_final.tsv", sep='\t', index=False)

### Running PySnail

In [ ]:
pysnail("in/qsmooth_in/RNA/xprs/RNA_xprs_final.tsv", "in/qsmooth_in/RNA/groups/RNA_groups_final.tsv", "out/qsmooth_out/RNA/output/RNA_qsmooth.tsv")

In [ ]:
RNA_qsmooth_df = pd.read_csv("out/qsmooth_out/RNA/output/RNA_qsmooth.tsv", sep='\t', header=[0, 1], index_col=0, low_memory=False)
RNA_qsmooth_df

In [ ]:
plotDens(RNA_qsmooth_df, xlabel="qsmooth RNA counts", title= "Density Plot for the First 50 Samples", amount=50)

In [ ]:
plotDens(RNA_qsmooth_df, xlabel="qsmooth RNA counts", title= "Density Plot for the First 200 Samples", amount=200)

In [ ]:
plotDens(RNA_qsmooth_df, xlabel="qsmooth RNA counts", title= "Density Plot for the All Samples", amount=4178)

### TFI Data

In [ ]:
TFI_raw_counts_df = pd.read_csv("in/raw_data/TFI/GSE59086_tfoe-full-with-replicates.csv").T
# Fix format of TFI data
TFI_raw_counts_df.columns = TFI_raw_counts_df.iloc[0]
TFI_raw_counts_df = TFI_raw_counts_df.drop(TFI_raw_counts_df.index[0])
TFI_raw_counts_df.columns = TFI_raw_counts_df.columns.str.replace('\n', '')
TFI_raw_counts_df = TFI_raw_counts_df.reset_index(drop=False).rename(columns={"index":"Geneid"})

In [ ]:
TFI_raw_counts_df

In [ ]:
plotDens(TFI_raw_counts_df, xlabel="log2 TFI counts", title= "Density Plot for the First 50 Samples", amount=50)

#### Extracting groups

In [ ]:
TFI_metadata_df = pd.read_csv("in/metadata/GSE59086_tfoe-metadata.tsv", sep='\t')

In [ ]:
TFI_metadata_df

In [ ]:
unique_values = TFI_metadata_df["TFI"].unique()
group_mapping = {value: idx + 1 for idx, value in enumerate(unique_values)}

# Apply the mapping to create the Group column
TFI_metadata_df["Group"] = TFI_metadata_df["TFI"].map(group_mapping)

In [ ]:
TFI_metadata_df[["Accession","Group"]]

In [ ]:
TFI_groups = TFI_metadata_df[["Accession","Group"]]

#### Writing Data for qsmooth

In [ ]:
TFI_raw_counts_df.to_csv("in/qsmooth_in/TFI/xprs/TFI_xprs.tsv", sep='\t', index=False)
TFI_groups.to_csv("in/qsmooth_in/TFI/groups/TFI_groups.tsv", sep='\t', index=False, header=None)

#### Running PySnail

In [ ]:
pysnail("in/qsmooth_in/TFI/xprs/TFI_xprs.tsv", "in/qsmooth_in/TFI/groups/TFI_groups.tsv")

In [ ]:
TFI_qsmooth_df = pd.read_csv("out/qsmooth_out/TFI/output/TFI_qsmooth.tsv", sep='\t', header=[0, 1], index_col=0, low_memory=False)
TFI_qsmooth_df

In [ ]:
plotDens(TFI_qsmooth_df, xlabel="qsmooth TFI counts", title= "Density Plot for the First 50 Samples", amount=50)

### Merging RNA + TFI

In [ ]:
RNA_values = TPM_log2_filtered_df2.iloc[:, 1:]
RNA_median = RNA_values.stack().median()
print("RNA median", RNA_median)
print("Min RNA value:", RNA_values.min().min())
print("Max RNA value:", RNA_values.max().max())

In [ ]:
TFI_values = TFI_raw_counts_df.iloc[:, 1:]
TFI_median = TFI_values.stack().median()
print("TFI median:", TFI_median)
print("Min TFI value:", TFI_values.min().min())
print("Max TFI value:", TFI_values.max().max())

In [ ]:
print("Difference between medians:", TFI_median - RNA_median)

In [ ]:
RNA_counts_added_df = TPM_log2_filtered_df2.iloc[:,1:] + 5.3

In [ ]:
RNA_counts_added_df

In [ ]:
RNA_median2 = RNA_counts_added_df.stack().median()
print("New RNA median", RNA_median2)
print("Min RNA value:", RNA_counts_added_df.min().min())
print("Max RNA value:", RNA_counts_added_df.max().max())

In [ ]:
RNA_counts_added_df = pd.concat([TPM_log2_filtered_df2["Geneid"], RNA_counts_added_df], axis=1)

Merge both the TPM Rna counts and TFI raw counts data frames together

In [ ]:
merged_df = RNA_counts_added_df.merge(TFI_raw_counts_df, on='Geneid', how='inner')

In [ ]:
merged_df

In [ ]:
merged_df = RNA_counts_added_df.merge(TFI_raw_counts_df, on='Geneid', how='inner')

Merge groups together after incrementing TFI

In [ ]:
TFI_groups_2 = TFI_groups.rename(columns={"Accession":"Sample"})

In [ ]:
TFI_groups_2["Group"] += 1537 # 1,537 was the number of the last RNA group

In [ ]:
TFI_groups_2

In [ ]:
merged_groups = pd.concat([RNA_groups_df, TFI_groups_2], axis=0)
merged_groups

#### Writing data for qsmooth

In [ ]:
merged_df.to_csv("in/qsmooth_in/merged/xprs/merged_xprs.tsv", sep='\t', index=False)
merged_groups.to_csv("in/qsmooth_in/merged/groups/merged_groups.tsv", sep='\t', index=False, header=None)

#### Running PySnail

In [ ]:
pysnail("in/qsmooth_in/merged/xprs/merged_xprs.tsv", "in/qsmooth_in/merged/groups/merged_groups.tsv", "out/qsmooth_out/merged/output/merged_qsmooth.tsv")

In [ ]:
merged_qsmooth_df = pd.read_csv("out/qsmooth_out/merged/output/merged_qsmooth.tsv", sep='\t', header=[0, 1], index_col=0, low_memory=False)
merged_qsmooth_df

Subtracting median for each sample

In [ ]:
medians = merged_qsmooth_df.median()
medians

In [ ]:
merged_adjusted_df = merged_qsmooth_df.subtract(medians, axis=1)
merged_adjusted_df

In [ ]:
plotDens(merged_adjusted_df, xlabel="qsmooth RNA + TFI after subtracting median per sample", title= "Density Plot for All Samples", amount=4876)

## Visualizations

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import umap
import umap.plot

### UMAP (RNA + TFI)

In [ ]:
from sklearn import cluster, mixture
from sklearn.neighbors import kneighbors_graph
from itertools import cycle, islice
import warnings
import time

In [ ]:
def umap_projection(df, save_path, plot_name='UMAP Projection', n_clusters=5):
    """
    Performs dimensionality reduction using UMAP, applies multiple clustering algorithms on the reduced data,
    and visualizes the results in a grid of scatter plots. The function standardizes the numeric data from the
    input DataFrame, reduces it to two dimensions using UMAP, and then applies several clustering algorithms,
    each generating a different clustering solution, which are visualized.

    Parameters:
    -----------
    df : pandas.DataFrame
        Input DataFrame containing numeric data for clustering. Non-numeric columns are excluded.
    
    save_path : str
        The file path where the resulting plot will be saved in SVG format.

    plot_name : str, optional (default='UMAP Projection')
        Title for the plot that will be displayed on top of the generated figure.

    n_clusters : int, optional (default=5)
        The number of clusters to use for algorithms that require specifying the number of clusters 
        (e.g., KMeans, Spectral Clustering).

    Returns:
    --------
    None
        The function generates a figure with scatter plots for each clustering algorithm and saves the plot 
        as an SVG file to the specified save path.

    Notes:
    ------
    - The function applies various clustering algorithms including MiniBatch KMeans, Affinity Propagation, 
      MeanShift, Spectral Clustering, Ward's method, DBSCAN, HDBSCAN, OPTICS, BIRCH, and Gaussian Mixture.
    - The colors of points in each subplot represent the clusters, and outliers (if any) are marked in black.
    - Execution time for each clustering algorithm is displayed on its respective subplot.
    """
    # Drop non-numeric columns
    dfT = df.T.reset_index()
    dfT.columns = ['Group'] + list(dfT.columns[1:])
    
    # Extract numeric data only
    numeric_cols = dfT.select_dtypes(include=[np.number]).columns
    train = dfT[numeric_cols]
    
    # Standardize the numeric data
    scaled_X = StandardScaler().fit_transform(train)

    # UMAP dimensionality reduction
    reducer = umap.UMAP()
    embedding = reducer.fit_transform(scaled_X)

    # Connectivity matrix for structured Ward (used in hierarchical clustering)
    connectivity = kneighbors_graph(embedding, n_neighbors=10, include_self=False)
    connectivity = 0.5 * (connectivity + connectivity.T)

    # Set up cluster parameters
    default_base = {
        "eps": 0.3,
        "min_samples": 7,
        "xi": 0.05,
        "min_cluster_size": 15,  # Update min_cluster_size to a valid int (e.g., 15)
        "random_state": 42,
    }

    # Clustering algorithms to apply
    clustering_algorithms = (
        ("MiniBatch KMeans", cluster.MiniBatchKMeans(n_clusters=n_clusters, random_state=42)),
        ("Affinity Propagation", cluster.AffinityPropagation(damping=0.9, random_state=42)),
        ("MeanShift", cluster.MeanShift(bin_seeding=True)),
        ("Spectral Clustering", cluster.SpectralClustering(n_clusters=n_clusters, affinity='nearest_neighbors', random_state=42)),
        ("Ward", cluster.AgglomerativeClustering(n_clusters=n_clusters, linkage="ward", connectivity=connectivity)),
        ("DBSCAN", cluster.DBSCAN(eps=default_base["eps"], min_samples=default_base["min_samples"])),
        ("HDBSCAN", cluster.HDBSCAN(min_samples=default_base["min_samples"], min_cluster_size=default_base["min_cluster_size"])),  # Valid int for min_cluster_size
        ("OPTICS", cluster.OPTICS(min_samples=default_base["min_samples"], xi=default_base["xi"], min_cluster_size=default_base["min_cluster_size"])),
        ("BIRCH", cluster.Birch(n_clusters=n_clusters)),
        ("Gaussian Mixture", mixture.GaussianMixture(n_components=n_clusters, covariance_type="full", random_state=42))
    )

    plt.figure(figsize=(15, 10))
    plt.suptitle(plot_name, fontsize=16)
    
    plot_num = 1
    for name, algorithm in clustering_algorithms:
        t0 = time.time()

        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=UserWarning)
            algorithm.fit(embedding)

        t1 = time.time()

        if hasattr(algorithm, "labels_"):
            y_pred = algorithm.labels_.astype(int)
        else:
            y_pred = algorithm.predict(embedding)

        # Create subplot for each algorithm
        plt.subplot(3, 4, plot_num)
        plt.title(name, fontsize=10)

        # Generate colors for clusters
        colors = np.array(
            list(islice(cycle(["#377eb8", "#ff7f00", "#4daf4a", "#f781bf", "#a65628", "#984ea3", "#999999", "#e41a1c", "#dede00"]), int(max(y_pred) + 1)))
        )
        # Add black color for outliers (if any)
        colors = np.append(colors, ["#000000"])
        plt.scatter(embedding[:, 0], embedding[:, 1], s=5, color=colors[y_pred])

        plt.xticks(())
        plt.yticks(())
        plt.text(0.99, 0.01, ("%.2fs" % (t1 - t0)).lstrip("0"), transform=plt.gca().transAxes, size=10, horizontalalignment="right")
        
        plot_num += 1

    plt.subplots_adjust(hspace=0.4, wspace=0.4)
    plt.savefig(save_path, format="svg")
    plt.show()

#### RNA

In [ ]:
umap_projection(RNA_qsmooth_df, "visualizations/RNA/RNA_qsmooth_UMAP.svg", plot_name="UMAP of RNA data")

#### TFI

In [ ]:
umap_projection(TFI_qsmooth_df, "visualizations/TFI/TFI_qsmooth_UMAP.svg", plot_name="UMAP of TFI data")

#### RNA + TFI

Grouping RNA data and TFI data seperately for visualization

In [ ]:
umap_merged_df = merged_adjusted_df
index_df = umap_merged_df.columns.to_frame(index=False)
index_df['Group'] = ['1' if i < 4178 else '2' for i in range(len(index_df))]
new_index = pd.MultiIndex.from_frame(index_df)
umap_merged_df.columns = new_index

In [ ]:
umap_projection(umap_merged_df, 'visualizations/merged/merged_qsmooth_UMAP.svg', plot_name='UMAP of RNA and TFI data')

In [ ]:
from sklearn.cluster import DBSCAN

In [ ]:
def umap_dbscan_projection(df, save_path, plot_name='UMAP DBSCAN Projection', eps=0.5, min_samples=3):
        """
    Performs dimensionality reduction using UMAP, applies the DBSCAN clustering algorithm on the reduced data,
    and visualizes the results. The function standardizes the numeric data from the input DataFrame, reduces it 
    to two dimensions using UMAP, and then applies DBSCAN clustering. The results are displayed in a scatter plot,
    and key statistics about the clustering are printed.

    Parameters:
    -----------
    df : pandas.DataFrame
        Input DataFrame containing numeric data for clustering. Non-numeric columns are excluded.
    
    save_path : str
        The file path where the resulting plot will be saved in SVG format.

    plot_name : str, optional (default='UMAP DBSCAN Projection')
        Title for the plot that will be displayed on top of the generated figure.

    eps : float, optional (default=0.5)
        The maximum distance between two samples for one to be considered as in the neighborhood of the other. 
        This is a key parameter for DBSCAN.

    min_samples : int, optional (default=3)
        The number of samples (or total weight) in a neighborhood for a point to be considered as a core point. 
        This includes the point itself.

    Returns:
    --------
    None
        The function generates a scatter plot of the UMAP embedding with DBSCAN cluster assignments and saves the plot
        as an SVG file to the specified save path. It also prints key clustering statistics, including:
        - Number of clusters (excluding outliers)
        - Number of outliers (points labeled as -1)
        - Size of the largest cluster (ignoring outliers)

    Notes:
    ------
    - Outliers are marked with black points, and each cluster is assigned a unique color.
    - The function prints the number of clusters found, the number of outliers (if any), and the size of the largest cluster.
    """
    # Drop non-numeric columns
    dfT = df.T.reset_index()
    dfT.columns = ['Group'] + list(dfT.columns[1:])
    
    # Extract numeric data only
    numeric_cols = dfT.select_dtypes(include=[np.number]).columns
    train = dfT[numeric_cols]
    
    # Standardize the numeric data
    scaled_X = StandardScaler().fit_transform(train)

    # UMAP dimensionality reduction
    reducer = umap.UMAP()
    embedding = reducer.fit_transform(scaled_X)

    # DBSCAN clustering
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning)
        dbscan.fit(embedding)
    
    # Get the cluster labels
    y_pred = dbscan.labels_.astype(int)

    # Get the number of clusters (excluding outliers labeled as -1)
    num_clusters = len(set(y_pred)) - (1 if -1 in y_pred else 0)
    
    # Get the number of outliers (points labeled as -1)
    num_outliers = np.sum(y_pred == -1)
    
    # Get the number of points in the largest cluster (ignoring outliers)
    largest_cluster_size = np.max(np.bincount(y_pred[y_pred != -1]))
  
    # Plot the DBSCAN clustering results
    plt.figure(figsize=(8, 8))
    plt.title(plot_name, fontsize=16)
    
    # Generate colors for clusters
    unique_labels = np.unique(y_pred)
    colors = np.array(
        list(islice(cycle(["#377eb8", "#ff7f00", "#4daf4a", "#f781bf", "#a65628", "#984ea3", "#999999", "#e41a1c", "#dede00"]), len(unique_labels)))
    )
    # Add black color for outliers (label -1)
    colors = np.append(colors, ["#000000"])
    
    # Scatter plot of UMAP embedding, colored by cluster labels
    plt.scatter(embedding[:, 0], embedding[:, 1], s=5, color=[colors[label] for label in y_pred])

    # Add UMAP axis labels
    plt.xlabel('UMAP 1')
    plt.ylabel('UMAP 2')

    plt.xticks(())
    plt.yticks(())
    
    # Save the plot
    plt.savefig(save_path, format='svg')
    plt.show()

    print(f"Number of clusters (excluding outliers): {num_clusters}")
    print(f"Number of outliers: {num_outliers}")
    print(f"Number of points in the largest cluster: {largest_cluster_size}")

In [ ]:
umap_dbscan_projection(RNA_qsmooth_df, 'visualizations/RNA/RNA_DBSCAN_UMAP.svg', plot_name='UMAP of RNA data')

In [ ]:
umap_dbscan_projection(TFI_qsmooth_df, 'visualizations/TFI/TFI_DBSCAN_UMAP.svg', plot_name='UMAP of TFI data', eps=0.225)

In [ ]:
dfT = umap_merged_df.T.reset_index()
dfT.columns = ['Group'] + list(dfT.columns[1:])

In [ ]:
dfT.T

In [ ]:
train = dfT.drop(columns=["Group","Sample"])

In [ ]:
train

In [ ]:
reducer = umap.UMAP()  # Creates a UMAP reducer object
scaled_X = StandardScaler().fit_transform(train)  # Standardizes the training data (z-scores)
embedding = reducer.fit_transform(scaled_X)  # Applies UMAP to reduce the dimensions of the scaled data

In [ ]:
z_scores = pd.DataFrame(scaled_X)

In [ ]:
z_scores

In [ ]:
# Sum up the values in each column
column_sums = z_scores.sum(axis=0)

# Convert the Series to a DataFrame with one row
summed_df = pd.DataFrame([column_sums])

In [ ]:
summed_df

In [ ]:
mapper = reducer.fit(scaled_X)

In [ ]:
umap.plot.points(mapper, background='white', show_legend=False)
plt.title("UMAP of RNA and TFI data (Standardized samples)", fontsize=12);
plt.xlabel(f'UMAP 1')
plt.ylabel(f'UMAP 2')

#### TFA MATRIX

In [ ]:
TFA_matrix_orig = pd.read_csv("visualizations/TFA/data/P_aggregate_new.csv", header=None)#.rename(columns={"Sample":"Geneid"})

In [ ]:
TFA_matrix_orig.columns = TPM_log2_filtered_df2.columns[1:]

In [ ]:
TFA_matrix_orig

In [ ]:
TFA_geneids = pd.read_csv("visualizations/TFA/data/tfa_matrix_geneids.csv").T.reset_index().rename(columns={"index":"Geneid"}).drop(214)

In [ ]:
TFA_geneids

In [ ]:
TFA_matrix_orig = pd.concat([TFA_geneids,TFA_matrix_orig], axis=1)

In [ ]:
TFA_matrix = TFA_matrix_orig[TFA_matrix_orig['Geneid'].isin(TPM_log2_filtered_df2['Geneid'])]

In [ ]:
TFA_matrix

In [ ]:
TFA_train = TFA_matrix.iloc[:,1:]

In [ ]:
TFA_train

In [ ]:
reducer = umap.UMAP()  # Creates a UMAP reducer object
scaled_X = StandardScaler().fit_transform(TFA_train).T  # Standardizes the training data (z-scores) across features 
# Applies UMAP to reduce the dimensions of the scaled data
mapper = reducer.fit(scaled_X)

umap.plot.points(mapper, background='white', show_legend=False)
plt.title("UMAP of TFA Matrix" , fontsize=12);
plt.xlabel(f'UMAP 1')
plt.ylabel(f'UMAP 2')

In [ ]:
umap_dbscan_projection(TFA_train, 'visualizations/TFA/TFA_DBSCAN_UMAP.svg', plot_name='UMAP of TFA Matrix')

### Clustering Aglorithms

In [ ]:
#!pip install hdbscan

In [ ]:
from sklearn.cluster import DBSCAN, OPTICS
import hdbscan

In [ ]:
def __embedding__(df):
    # Drop non-numeric columns
    dfT = df.T.reset_index()
    dfT.columns = ['Group'] + list(dfT.columns[1:])
    
    # Extract numeric data only
    numeric_cols = dfT.select_dtypes(include=[np.number]).columns
    train = dfT[numeric_cols]
    
    # Standardize the numeric data
    scaled_X = StandardScaler().fit_transform(train)

    reducer = umap.UMAP()
    embedding = reducer.fit_transform(scaled_X)

    return embedding

In [ ]:
datasets = {
    "RNA": __embedding__(RNA_qsmooth_df),
    "TFI": __embedding__(TFI_qsmooth_df),
    "TFA":__embedding__(TFA_train)
}

In [ ]:
import time
start = time.time()

In [ ]:
# Function to calculate cluster metrics
def get_cluster_metrics(labels):
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_outliers = list(labels).count(-1)
    max_cluster_size = max([list(labels).count(i) for i in set(labels) if i != -1], default=0)
    return n_clusters, n_outliers, max_cluster_size

results = []

In [ ]:
# DBSCAN loop over epsilon (log scale)
eps_values = np.logspace(-1, 1, 50)

for dataset_name, data in datasets.items():
    for eps in eps_values:
        dbscan = DBSCAN(eps=eps, min_samples=3)
        labels = dbscan.fit_predict(data)
        n_clusters, n_outliers, max_cluster_size = get_cluster_metrics(labels)
        results.append({
            "Algorithm": "DBSCAN",
            "Dataset": dataset_name,
            "eps": eps,
            "min_samples": 3,
            "n_clusters": n_clusters,
            "n_outliers": n_outliers,
            "max_cluster_size": max_cluster_size
        })

In [ ]:
# HDBSCAN (run once per dataset)
for dataset_name, data in datasets.items():
    hdbscan_clusterer = hdbscan.HDBSCAN(min_samples=3)
    labels = hdbscan_clusterer.fit_predict(data)
    n_clusters, n_outliers, max_cluster_size = get_cluster_metrics(labels)
    results.append({
        "Algorithm": "HDBSCAN",
        "Dataset": dataset_name,
        "min_samples": 3,
        "n_clusters": n_clusters,
        "n_outliers": n_outliers,
        "max_cluster_size": max_cluster_size
    })

In [ ]:
# OPTICS loop over min_samples from 2 to 6 (was 1 to 5 but 2 is the lowest integer value OPTICS takes for min_samples)
for dataset_name, data in datasets.items():
    for min_samples in range(2, 7):
        optics = OPTICS(min_samples=min_samples)
        labels = optics.fit_predict(data)
        n_clusters, n_outliers, max_cluster_size = get_cluster_metrics(labels)
        results.append({
            "Algorithm": "OPTICS",
            "Dataset": dataset_name,
            "min_samples": min_samples,
            "n_clusters": n_clusters,
            "n_outliers": n_outliers,
            "max_cluster_size": max_cluster_size
        })

In [ ]:
end = time.time()
print("Time elasped:", end - start, "seconds")

In [ ]:
results_df = pd.DataFrame(results)

In [ ]:
results_df

In [ ]:
def dbscan_hyperopt_plots(dataset):
    dbscan_df = results_df[(results_df['Algorithm'] == 'DBSCAN') & (results_df['Dataset'] == dataset)]
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # First plot: Total clusters
    axes[0].plot(dbscan_df['eps'], dbscan_df['n_clusters'], marker='x', linestyle='-', color='blue')
    axes[0].set_xscale('log')
    axes[0].set_xlabel('Epsilon [log scale]')
    axes[0].set_ylabel('No. of Clusters')

    # Second plot: Largest cluster
    axes[1].plot(dbscan_df['eps'], dbscan_df['max_cluster_size'], marker='x', linestyle='-', color='orange')
    axes[1].set_xscale('log')
    axes[1].set_xlabel('Epsilon [log scale]')
    axes[1].set_ylabel('No. of Samples in Largest Cluster')

    # Third plot: Number of outliers
    axes[2].plot(dbscan_df['eps'], dbscan_df['n_outliers'], marker='x', linestyle='-', color='green')
    axes[2].set_xscale('log')
    axes[2].set_xlabel('Epsilon [log scale]')
    axes[2].set_ylabel('No. of Outliers')

    fig.suptitle('Epsilon Hyperparameterization for DBSCAN on ' + dataset + ' Dataset', fontsize=16)
    plt.tight_layout()

    # Display the plot
    plt.show()

In [ ]:
dbscan_hyperopt_plots("RNA")

In [ ]:
dbscan_hyperopt_plots("TFI")

In [ ]:
dbscan_hyperopt_plots("TFA")

In [ ]:
results_df.to_csv('clustering_results.csv', index=False)

### Histogram (Log10 RNA Counts)

Here we'll build a histogram of our orignial RNA counts before we filtered the 444 low quality samples.

In [ ]:
RNA_hist_data = RNA_counts_df.drop(columns="Geneid")

In [ ]:
RNA_hist_data

In [ ]:
sum_df = pd.DataFrame(RNA_hist_data.sum()).reset_index().rename(columns={0:"Counts", "index":"Sample"})

In [ ]:
sum_df

In [ ]:
my_value = 0
results = sum_df.loc[sum_df["Counts"] == my_value]
print(results)

In [ ]:
sum_df_log10 = np.log10(sum_df["Counts"] + 1)

In [ ]:
sum_df_log10

In [ ]:
line_value = 5.6
plt.figure(figsize=(8, 8))
sns.histplot(data=sum_df_log10, legend=False, bins=50)
plt.axvline(line_value, color='red', linestyle='dashed', linewidth=2)
plt.xlabel("RNA counts")
ticks = np.arange(0, int(sum_df_log10.max()) + 1, 1)
plt.xticks(ticks)
plt.savefig("visualizations/RNA/RNA_hist.svg", format='svg')
plt.show()

### Histogram (filtered data)

In [ ]:
hist_data_filtered = RNA_hist_data.drop(columns=samples_to_exclude)

In [ ]:
hist_data_filtered

In [ ]:
filtered_sum_df = pd.DataFrame(hist_data_filtered.sum()).reset_index().rename(columns={0:"Counts", "index":"Sample"})

In [ ]:
filtered_sum_df

In [ ]:
filtered_sum_log10_df = np.log10(filtered_sum_df["Counts"] + 1)

In [ ]:
sns.histplot(data=filtered_sum_log10_df, legend=False, bins=50)

In [ ]:
line_value = 5.6
sns.histplot(data=filtered_sum_log10_df, legend=False, bins=50)
plt.axvline(line_value, color='red', linestyle='dashed', linewidth=2)
plt.xlabel("filtered RNA counts")
# Show the plot
plt.show()

### Histograms

In [ ]:
TFExpvsActCorrHist_df = pd.read_excel("visualizations/hist_data/TFExpvsActCorrHist.xlsx")
TFExpvsActCorrHist_df.head()

In [ ]:
# Extracting the data for histogram
bin_centers = TFExpvsActCorrHist_df['Bin Center']
counts = TFExpvsActCorrHist_df['Count']

In [ ]:
# Creating the histogram
plt.bar(bin_centers, counts, width=0.1, edgecolor='black')
plt.xlabel('Bin Center')
plt.ylabel('Count')
plt.title('Histogram of TFExp vs ActCorr')
plt.savefig("visualizations/hist_data/TFExpvsActCorrHist.svg")
plt.show()

In [ ]:
xls = pd.ExcelFile("visualizations/hist_data/TFI TFA Percentiles.xlsx")
xls.sheet_names

In [ ]:
tfa_percentiles_df = pd.read_excel(xls, sheet_name='TFA Percentiles')
histogram_counts_df = pd.read_excel(xls, sheet_name='Histogram Counts')

In [ ]:
tfa_percentiles_df.head()

In [ ]:
histogram_counts_df.head()

In [ ]:
# Extracting the necessary data from the 'Histogram Counts' sheet
bin_centers = histogram_counts_df['Bin Center']
TFI_net_counts = histogram_counts_df['TFI Net']

# Creating a bar plot of the histogram counts for 'TFI Net'
plt.figure(figsize=(10, 6))
plt.bar(bin_centers, TFI_net_counts, width=0.05, color='blue', alpha=0.7)

# Setting titles and labels
plt.title('Histogram of TFI Net Counts', fontsize=14)
plt.xlabel('Bin Center', fontsize=12)
plt.ylabel('Count', fontsize=12)

# Show the plot
plt.grid(True)
plt.show()

In [ ]:
exp_vs_activity_xls = pd.ExcelFile("visualizations/hist_data/table_exp_vs_activity.xlsx")

# Check the sheet names to understand the structure of this file
exp_vs_activity_xls.sheet_names

In [ ]:
# Load data from the relevant sheet to inspect it, assuming Pearson's r might be in 'RNAseq Compendium'
rna_seq_df = pd.read_excel(exp_vs_activity_xls, sheet_name='RNAseq Compendium')

# Check the first few rows to identify the "Pearson's r" column
rna_seq_df.head()

In [ ]:
# Load the other sheets to search for Pearson's r
TFI_compendium_df = pd.read_excel(exp_vs_activity_xls, sheet_name='TFI Compendium')
rna_seq_tfas_df = pd.read_excel(exp_vs_activity_xls, sheet_name='RNAseq TFAs')

# Display first few rows to check for "Pearson's r" column
TFI_compendium_df.head(), rna_seq_tfas_df.head()

In [ ]:
# The "Pearson's r" column is in the 'RNAseq TFAs' sheet, we will create a histogram for it.

pearsons_r_values = rna_seq_tfas_df['Pearson r']

# Create the histogram for the Pearson's r values
plt.figure(figsize=(10, 6))
plt.hist(pearsons_r_values, bins=20, color='green', alpha=0.7)

# Set titles and labels
plt.title("Histogram of Pearson's r Values (TF Expression vs TFA)", fontsize=14)
plt.xlabel("Pearson's r", fontsize=12)
plt.ylabel("Frequency", fontsize=12)

# Show the plot
plt.grid(True)
plt.show()

### Median vs MAD (For all ~4k genes for both RNA + TFI)

In [ ]:
from scipy.stats import gaussian_kde
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import plotly.express as px

In [ ]:
def plot_MEDvsMAD(df, plot_title, save_fig_path):
    # Calculate median and MAD
    x = df.median(axis=1)
    y = df.sub(x, axis=0).abs().median(axis=1)
    
    # Calculate the point density
    xy = np.vstack([x, y])
    z = gaussian_kde(xy)(xy)
    
    # Create the plot
    fig = px.scatter(
        x=x,
        y=y,
        color=z,
        color_continuous_scale='Viridis',
        labels={'x': 'Median Counts (qsmooth)', 'y': 'MAD Counts (qsmooth)', 'color': 'Density'},
        title=plot_title
    )
    return fig

In [ ]:
fig = plot_MEDvsMAD(RNA_qsmooth_df, "Median vs MAD", 'visualizations/RNA/RNA_MEDvsMAD_NEW.png')
fig.show()

In [ ]:
def plot_MEDvsMAD(df, plot_title, save_fig_path):
    # Calculate median and MAD
    x = df.median(axis=1)
    y = df.sub(x, axis=0).abs().median(axis=1)
    
    # Calculate the point density
    xy = np.vstack([x, y])
    z = gaussian_kde(xy)(xy)
    
    # Create the plot
    fig, ax = plt.subplots(figsize=(8, 8))
    scatter = ax.scatter(x, y, c=z, s=15)

    # Add labels and title
    ax.set_xlabel('Median Counts')
    ax.set_ylabel('Median Absolute Deviation')
    ax.set_title(plot_title)
    
    # Add odd and even tick marks on the x-axis
    x_min, x_max = int(np.floor(min(x))), float(np.ceil(max(x)))
    ticks = np.arange(x_min, x_max + 1, 1)  # Generate tick marks for both odd and even numbers
    plt.xticks(ticks)  # Apply the tick marks

    # Save the figure
    plt.savefig(save_fig_path)

    # Show the plot
    plt.show()

In [ ]:
plot_MEDvsMAD(RNA_qsmooth_df, 'Median vs MAD plot of RNA', 'visualizations/RNA/RNA_MEDvsMAD.svg')

In [ ]:
plot_MEDvsMAD(TFI_qsmooth_df, 'Median vs MAD plot of TFI', 'visualizations/TFI/TFI_MEDvsMAD.svg')

In [ ]:
def plot_MEDvsMAD_TFA(df, plot_title, save_fig_path):
    
    # Calculate median and MAD (Median Absolute Deviation)
    x = df.median(axis=1)
    y = df.sub(x, axis=0).abs().median(axis=1)
    
    # Calculate the point density
    xy = np.vstack([x, y])
    z = gaussian_kde(xy)(xy)
    
    # Create the plot
    fig, ax = plt.subplots(figsize=(8, 8))
    scatter = ax.scatter(x, y, c=z, s=15)

    # Add labels and title
    ax.set_xlabel('Median Counts')
    ax.set_ylabel('Median Absolute Deviation')
    ax.set_title(plot_title)
    
    # Save the figure
    plt.savefig(save_fig_path)  # Fixed the typo

    # Show the plot
    plt.show()

In [ ]:
plot_MEDvsMAD_TFA(TFA_train, "Median vs MAD of TFA Matrix", "visualizations/TFA/TFA_Matrix_MedvsMAD.svg")

#### TFA Expression data Med vs MAD

In [ ]:
TFA_xprs_df = pd.read_csv("qsmooth/RNA/output/RNA_qsmooth.tsv", sep='\t', header=1, low_memory=False).drop(0).rename(columns={"Sample":"Geneid"})

In [ ]:
TFA_xprs_df = TFA_xprs_df[TFA_xprs_df['Geneid'].isin(TFA_matrix['Geneid'])].reset_index(drop=True)

In [ ]:
TFA_xprs_df

In [ ]:
#TFA_xprs_df.to_csv("qsmooth/RNA/output/TFA_xprs/TFA_xprs.csv")

In [ ]:
plot_MEDvsMAD_TFA(TFA_xprs_df.iloc[:,1:], "Median vs MAD of TFA Expression Data", "visualizations/TFA/TFA_xprs_MedvsMAD.svg")

### TRN Matrix

In [ ]:
TRN_matrix_df = pd.read_table("raw_data/TRN_matrix/aggregate-mtb_candidate_v2.txt")

In [ ]:
TRN_matrix_df

In [ ]:
df_reset = TRN_matrix_df.reset_index(drop=True)

In [ ]:
new_header = df_reset.columns.tolist()

In [ ]:
header_df = pd.DataFrame([new_header], columns=new_header)

In [ ]:
df_with_header_row = pd.concat([header_df, df_reset], ignore_index=True)

In [ ]:
TRN_matrix_df2 = df_with_header_row.rename(columns={"Rv2989":"TF", "Rv2988c":"Gene", "255113.0":"index", "up":"State"}).drop(columns="index")

In [ ]:
TRN_matrix_df2

In [ ]:
# Convert 'up' and 'down' to 1 and -1
TRN_matrix_df2['State'] = TRN_matrix_df2['State'].map({'up': 1, 'down': 0})

In [ ]:
TRN_matrix_df2

In [ ]:
unique_values = TRN_matrix_df2['TF'].unique()
count = 0
for value in unique_values:
    count += 1
print(count)

In [ ]:
unique_values = TRN_matrix_df2['Gene'].unique()
count = 0
for value in unique_values:
    count += 1
print(count)

In [ ]:
unique_values = RNA_metadata_df['BioProject'].unique()
count = 0
for value in unique_values:
    count += 1
print(count)

In [ ]:
heatmap_data = TRN_matrix_df2.pivot(index='TF', columns='Gene', values='State')

In [ ]:
fig = px.imshow(heatmap_data, color_continuous_scale='Viridis')

# Update the layout to remove labels
fig.update_layout(
    title='Heatmap of Transcription Factors and Genes States',
    xaxis=dict(title='Genes', showticklabels=False),
    yaxis=dict(title='Transcription Factors', showticklabels=False)
)

 ### No. of samples from SRA (RNA only)

In [ ]:
samples_in_counts = RNA_counts_df.columns
filtered_metadata = RNA_metadata_df[RNA_metadata_df['Run'].isin(samples_in_counts)]

In [ ]:
filtered_metadata

In [ ]:
filtered_metadata['Year'] = pd.to_datetime(filtered_metadata['ReleaseDate']).dt.year

In [ ]:
result_df = filtered_metadata[['Run', 'Year']]

In [ ]:
result_df

In [ ]:
# Step 1: Group the data by year and count the number of samples for each year
samples_by_year = result_df.groupby('Year').size().reset_index(name='Count')

# Step 2: Calculate the cumulative sum of the counts
samples_by_year['Number of Publicly Available Samples'] = samples_by_year['Count'].cumsum()

In [ ]:
samples_by_year[["Year", "Number of Publicly Available Samples"]]

In [ ]:
#!pip install plotly

In [ ]:
fig = px.area(samples_by_year, x='Year', y='Number of Publicly Available Samples')

fig.show()

In [ ]:
# Assuming you have a DataFrame called samples_by_year
# Create the area plot
fig, ax = plt.subplots(figsize=(10, 10))

# Plot the area chart
samples_by_year.plot(
    x='Year', 
    y='Number of Publicly Available Samples', 
    kind='area', 
    ax=ax,
    color='steelblue',
    legend=False
)

# Highlight the trendline
ax.plot(samples_by_year['Year'], samples_by_year['Number of Publicly Available Samples'], color='darkblue', linewidth=2)

# Set x-axis ticks to show all years
ax.set_xticks(samples_by_year['Year'])  # Set the ticks
ax.set_xticklabels(samples_by_year['Year'], rotation=45)  # Rotate labels for better visibility

# Set x-axis limits to remove the whitespace at the end
ax.set_xlim(samples_by_year['Year'].min(), samples_by_year['Year'].max())

# Add labels and title
ax.set_xlabel('Year')
ax.set_ylabel('Total Samples')
ax.set_title('Number of Publicly Available Samples by Year')

# Set the aspect ratio to be square
ax.set_aspect('auto')

# Show the plot
plt.savefig("visualizations/RNA/samples_per_year.svg")
plt.show()

### Metadata summary

#### Filtering metadata
First we filter our metadata table to only include the samples that are included in our final RNA compendium

In [ ]:
run_values = TPM_log2_filtered_df2.columns

# Filter RNA_metadata_df where the 'Run' column matches any of the values in run_values
filtered_RNA_metadata_df = RNA_metadata_df[RNA_metadata_df['Run'].isin(run_values)]

# Display the filtered DataFrame
filtered_RNA_metadata_df

#### Filtering bioprojects

In [ ]:
project_ids = [
    "PRJNA507986", "PRJNA613916", "PRJNA645312", "PRJNA645886", "PRJNA667658", 
    "PRJNA675843", "PRJNA694989", "PRJNA695291", "PRJNA698563", "PRJNA251986", 
    "PRJNA699888", "PRJNA701064", "PRJNA701251", "PRJNA701434", "PRJNA701584", 
    "PRJNA707587", "PRJNA733783", "PRJNA263293", "PRJNA276810", "PRJNA299665", 
    "PRJNA310781", "PRJNA327080", "PRJNA353497", "PRJNA360809", "PRJNA376285", 
    "PRJNA378382", "PRJNA175135", "PRJNA390669", "PRJNA399098", "PRJNA421561", 
    "PRJNA433880", "PRJNA436144", "PRJNA453604", "PRJNA478238", "PRJNA478245", 
    "PRJNA478391", "PRJNA480455", "PRJNA480466", "PRJNA484003", "PRJNA484017", 
    "PRJNA484020", "PRJNA484385", "PRJNA484402", "PRJNA484403", "PRJNA484405", 
    "PRJNA484824", "PRJNA487629", "PRJNA488113", "PRJNA488546", "PRJNA489616", 
    "PRJNA489615", "PRJNA489617", "PRJNA507615", "PRJNA508198", "PRJNA511673", 
    "PRJNA521480", "PRJNA208118", "PRJNA219730", "PRJNA287328", "PRJNA314689", 
    "PRJNA323606", "PRJNA353127", "PRJNA382447", "PRJNA400374", "PRJNA419075", 
    "PRJNA422199", "PRJNA429748", "PRJNA433582", "PRJNA476854", "PRJNA488900", 
    "PRJNA509380", "PRJNA525595", "PRJNA527616", "PRJNA586773", "PRJNA601528", 
    "PRJNA627603", "PRJNA631850", "PRJNA633785", "PRJNA649164", "PRJNA660032", 
    "PRJNA664110", "PRJNA698254", "PRJNA716254", "PRJNA720822", "PRJNA777953", 
    "PRJNA820116", "PRJNA838962", "PRJNA295556", "PRJNA313774", "PRJNA679238", 
    "PRJNA686956", "PRJNA694147", "PRJNA713504", "PRJNA830343", "PRJNA886436"]

In [ ]:
filtered_bioprojects = filtered_RNA_metadata_df[~filtered_RNA_metadata_df['BioProject'].isin(project_ids)]

In [ ]:
unique_bioprojects = filtered_bioprojects['BioProject'].unique()

In [ ]:
bioprojects_df = pd.DataFrame(unique_bioprojects).rename(columns={0:"BioProject"}).dropna()

In [ ]:
bioprojects_df

In [ ]:
bioprojects_df['Link'] = bioprojects_df['BioProject'].apply(
    lambda x: f'https://www.ncbi.nlm.nih.gov/bioproject/?term={x}'
)

In [ ]:
bioprojects_df

In [ ]:
def print_matching_runs(bioproject):
    matching_rows = filtered_RNA_metadata_df[filtered_RNA_metadata_df['BioProject'] == bioproject]
    matching_runs = matching_rows['Run'].tolist()
    
    if matching_runs:
        print(f"Run values matching BioProject '{bioproject}':")
        for run in matching_runs:
            print(run)
    else:
        print(f"No matching Run values found for BioProject '{bioproject}'.")

In [ ]:
bioproject_list = bioprojects_df['BioProject'].tolist()

In [ ]:
for bioproject in bioproject_list:
    print_matching_runs(bioproject)
    print("--------------------------------")

#### Writing bioproject data

In [ ]:
#bioprojects_df.to_csv('visualizations/RNA/metadata_summary/bioprojects/new/unique_bioprojects.csv', index=False)